In [2]:
from sympy import *
from scipy.integrate import solve_ivp
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# --- Physical Constants ---
R_E = 6378137.0         # Radius of Earth (meters)
g0 = 9.80665            # Standard gravity at sea level (m/s^2)

# --- Atmospheric & Aerodynamic Assumptions ---
rho_0 = 1.225           # Sea-level atmospheric density (kg/m^3)
H_scale = 7500.0        # Atmospheric scale height (meters)
mass = 5000.0           # Spacecraft mass (kg)
area = 10.0             # Reference frontal area (m^2)
C_D = 1.5               # Baseline drag coefficient
C_L = 0.5               # Baseline lift coefficient

In [4]:
def get_density(h):
    """Calculates atmospheric density using an exponential model assumption."""
    return rho_0 * np.exp(-h / H_scale) if h >= 0 else rho_0

def calc_forces_and_load(h, v):
    """Calculates lift, drag, and the total load factor in Earth g's."""
    rho = get_density(h)
    q = 0.5 * rho * (v**2)  # Dynamic pressure
    
    lift = q * C_L * area
    drag = q * C_D * area
    load_g = np.sqrt(lift**2 + drag**2) / (mass * g0)
    
    return lift, drag, load_g

In [5]:
def entry_equations_of_motion(t, state):
    """
    Defines the 4 ODE derivatives for the state vector: [h, v, gamma, s]
    where s is the downrange distance traveled over the Earth's surface.
    """
    h, v, gamma, s = state
    
    # Fetch forces
    lift, drag, _ = calc_forces_and_load(h, v)
    g = g0 * (R_E / (R_E + h))**2  # Gravity variation with altitude
    
    # 1. Altitude derivative
    dh_dt = v * np.sin(gamma)
    
    # 2. Velocity derivative
    dv_dt = -(drag / mass) - g * np.sin(gamma)
    
    # 3. Flight-path angle derivative
    dgamma_dt = (lift / (mass * v)) + ((v / (R_E + h)) - (g / v)) * np.cos(gamma)
    
    # 4. Downrange distance derivative (The 4th State)
    ds_dt = (R_E / (R_E + h)) * v * np.cos(gamma)
    
    return [dh_dt, dv_dt, dgamma_dt, ds_dt]

In [6]:
# --- Initial Value Problem (IVP) Conditions ---
h_0 = 120000.0          # Entry interface altitude (120 km)
v_0 = 11200.0           # Escape velocity entry speed (m/s)
gamma_0 = np.radians(-6.0) # Entry angle converted to radians
s_0 = 0.0               # Initial downrange distance (meters)

initial_state = [h_0, v_0, gamma_0, s_0]
t_span = (0, 2000)      # Max simulation window (seconds)

In [7]:
def crash_event(t, state):
    """Triggers if the spacecraft hits the surface (altitude <= 0)."""
    return state[0]  # returns altitude h
crash_event.terminal = True
crash_event.direction = -1

def skip_event(t, state):
    """Triggers if the spacecraft bounces back out into deep space (h >= 150km after dipping)."""
    # Only active if time has progressed past the initial entry second
    return state[0] - 150000.0 if t > 10 else 1.0
skip_event.terminal = True
skip_event.direction = 1

# List of tracking thresholds to pass to scipy's solve_ivp
event_thresholds = [crash_event, skip_event]

In [18]:
# START OF MY CODE 
#
#
# 
import numpy as np
import csv

# Speeds to investigate, in m/s
entry_speeds = [10500.0, 11000.0, 11500.0]

# Initial flight-path angles, in degrees
entry_angles = np.linspace(-10.0, -1.0, 91)

# Numerical settings
dt = 0.025                  # Time step, seconds
angle_tolerance = 0.001     # Boundary bracket width, degrees

# Study criteria
skip_altitude = 150000.0    # Upward crossing altitude, meters
load_limit = 9.0            # Maximum aerodynamic load, g

In [19]:
def run_one_case(v_start, gamma_start_deg, dt):
    """
    Simulate one entry using forward Euler.

    State order matches the parameters notebook:
    [altitude, speed, flight-path angle, downrange distance].
    """

    state = np.array([
        h_0,
        v_start,
        np.radians(gamma_start_deg),
        s_0
    ], dtype=float)

    t = float(t_span[0])
    t_final = float(t_span[1])

    max_load_g = 0.0
    min_altitude = state[0]
    has_descended = False
    outcome = "unresolved"

    while True:
        h, v, gamma, s = state

        if not np.all(np.isfinite(state)) or v <= 0:
            outcome = "numerical failure"
            break

        _, _, load_g = calc_forces_and_load(h, v)

        max_load_g = max(max_load_g, float(load_g))
        min_altitude = min(min_altitude, float(h))

        if h < h_0:
            has_descended = True

        # Classify a skip after descent and upward passage of 150 km.
        if has_descended and h >= skip_altitude and gamma > 0:
            outcome = "skip"
            break

        if h <= 0:
            outcome = "surface"
            break

        if t >= t_final:
            break

        # Forward Euler step
        step = min(dt, t_final - t)

        derivatives = np.asarray(
            entry_equations_of_motion(t, state),
            dtype=float
        )

        state = state + step * derivatives
        t = t + step

    return {
        "v_start": float(v_start),
        "gamma_start_deg": float(gamma_start_deg),
        "outcome": outcome,
        "max_load_g": float(max_load_g),
        "min_altitude_m": float(min_altitude),
        "end_time_s": float(t)
    }

In [20]:
results = []

for speed in entry_speeds:
    print(f"Running speed: {speed / 1000:.2f} km/s")

    for angle in entry_angles:
        case = run_one_case(speed, angle, dt)
        results.append(case)

# Save results for the separate plotting notebook
with open("sweep_results.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print(f"\nFinished: {len(results)} trajectories.")
print("Saved sweep_results.csv")

Running speed: 10.50 km/s
Running speed: 11.00 km/s
Running speed: 11.50 km/s

Finished: 273 trajectories.
Saved sweep_results.csv


In [21]:
def refine_boundary(speed, angle_a, angle_b, boundary_type):
    """Refine a detected boundary by repeatedly halving its bracket."""

    def category(case):
        if boundary_type == "skip":
            if case["outcome"] not in ("skip", "surface"):
                return None
            return case["outcome"] == "skip"

        if boundary_type == "load":
            if case["outcome"] != "skip":
                return None
            return case["max_load_g"] <= load_limit

        raise ValueError("Unknown boundary type")

    case_a = run_one_case(speed, angle_a, dt)
    case_b = run_one_case(speed, angle_b, dt)

    category_a = category(case_a)
    category_b = category(case_b)

    if (
        category_a is None
        or category_b is None
        or category_a == category_b
    ):
        return None

    while abs(angle_b - angle_a) > angle_tolerance:
        angle_mid = 0.5 * (angle_a + angle_b)
        case_mid = run_one_case(speed, angle_mid, dt)
        category_mid = category(case_mid)

        if category_mid is None:
            return None

        if category_mid == category_a:
            angle_a = angle_mid
            case_a = case_mid
        else:
            angle_b = angle_mid
            case_b = case_mid

    return {
        "v_start": float(speed),
        "boundary_type": boundary_type,
        "angle_a_deg": float(angle_a),
        "angle_b_deg": float(angle_b),
        "outcome_a": case_a["outcome"],
        "outcome_b": case_b["outcome"],
        "load_a_g": float(case_a["max_load_g"]),
        "load_b_g": float(case_b["max_load_g"]),
        "dt_s": float(dt)
    }


boundary_results = []

for speed in entry_speeds:
    speed_cases = sorted(
        [case for case in results if case["v_start"] == speed],
        key=lambda case: case["gamma_start_deg"]
    )

    for case_a, case_b in zip(speed_cases[:-1], speed_cases[1:]):
        angle_a = case_a["gamma_start_deg"]
        angle_b = case_b["gamma_start_deg"]

        # Skip-to-surface boundary
        if {case_a["outcome"], case_b["outcome"]} == {"skip", "surface"}:
            boundary = refine_boundary(speed, angle_a, angle_b, "skip")

            if boundary is not None:
                boundary_results.append(boundary)

        # 9 g boundary, only when both cases are skips
        if case_a["outcome"] == "skip" and case_b["outcome"] == "skip":
            below_a = case_a["max_load_g"] <= load_limit
            below_b = case_b["max_load_g"] <= load_limit

            if below_a != below_b:
                boundary = refine_boundary(speed, angle_a, angle_b, "load")

                if boundary is not None:
                    boundary_results.append(boundary)


# Always write a header, even if no boundaries are found.
boundary_fields = [
    "v_start", "boundary_type", "angle_a_deg", "angle_b_deg",
    "outcome_a", "outcome_b", "load_a_g", "load_b_g", "dt_s"
]

with open("boundary_results.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=boundary_fields)
    writer.writeheader()
    writer.writerows(boundary_results)


# Display refined boundaries
for speed in entry_speeds:
    print(f"\nInitial speed: {speed / 1000:.2f} km/s")

    found = [
        item for item in boundary_results
        if item["v_start"] == speed
    ]

    if not found:
        print("No boundary found in the sampled range.")
    else:
        for item in found:
            print(
                f"{item['boundary_type']} boundary: "
                f"{item['angle_a_deg']:.4f}° to "
                f"{item['angle_b_deg']:.4f}°"
            )

print("\nSaved boundary_results.csv")


Initial speed: 10.50 km/s
skip boundary: -6.4086° to -6.4078°

Initial speed: 11.00 km/s
skip boundary: -7.0992° to -7.0984°
load boundary: -6.8117° to -6.8109°

Initial speed: 11.50 km/s
skip boundary: -7.7539° to -7.7531°
load boundary: -6.8672° to -6.8664°

Saved boundary_results.csv
